In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_score, recall_score, f1_score
import requests
from PIL import Image
import io
import os

tensorflow_available = False
try:
    from tensorflow.keras.applications import ResNet50, preprocess_input
    from tensorflow.keras.models import Model
    from tensorflow.keras.preprocessing import image
    tensorflow_available = True
except Exception as e:
    print("TensorFlow import failed:", e)
    print("Image feature extraction will be disabled. To enable TensorFlow, install the Microsoft C++ Redistributable for Visual Studio 2015-2019 and ensure TensorFlow is compatible with your Python version.")

print("Libraries imported successfully. TensorFlow available:", tensorflow_available)

ImportError: Could not find the DLL(s) 'msvcp140_1.dll'. TensorFlow requires that these DLLs be installed in a directory that is named in your %PATH% environment variable. You may install these DLLs by downloading "Microsoft C++ Redistributable for Visual Studio 2015, 2017 and 2019" for your platform from this URL: https://support.microsoft.com/help/2977003/the-latest-supported-visual-c-downloads

In [2]:
y= 10
print("Value of y:", y)

Value of y: 10


In [ ]:
# load the dataset
data = pd.read_csv('moviedataset.csv')

In [34]:
data.head()

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811


In [35]:
data.describe()

,id,popularity,vote_average,vote_count
count,10000.000000,10000.000000,10000.000000,10000.000000
mean,161243.505000,34.697267,6.621150,1547.309400
std,211422.046043,211.684175,0.766231,2648.295789
min,5.000000,0.600000,4.600000,200.000000
25%,10127.750000,9.154750,6.100000,315.000000
50%,30002.500000,13.637500,6.600000,583.500000
75%,310133.500000,25.651250,7.200000,1460.000000
max,934761.000000,10436.917000,8.700000,31917.000000


In [36]:
data.isnull().sum()

id                    0
title                 0
genre                 3
original_language     0
overview             13
popularity            0
release_date          0
vote_average          0
vote_count            0
dtype: int64

In [37]:
# TMDB API setup
api_key = '72d6eb78b2ce09c9ee018faaf3e54782'

BASE_URL = 'https://api.themoviedb.org/3'
IMAGE_BASE_URL = 'https://image.tmdb.org/t/p/w500'

In [38]:
# Download movie posters (limited to first 100 for testing)
os.makedirs('posters', exist_ok=True)

def fetch_poster(movie_id):
    url = f"{BASE_URL}/movie/{movie_id}?api_key={api_key}"
    response = requests.get(url)
    data = response.json()
    poster_path = data.get('poster_path')

    if poster_path:
        full_path = IMAGE_BASE_URL + poster_path
        img_response = requests.get(full_path)
        if img_response.status_code == 200:
            file_path = f"posters/{movie_id}.jpg"
            with open(file_path, 'wb') as handler:
                handler.write(img_response.content)
            return file_path
    return None

# Download posters for first 10 movies only (for testing)
poster_paths = []
for movie_id in data['id'].head(10):  # Limit to first 10 movies
    path = fetch_poster(movie_id)
    poster_paths.append(path)
data['poster_path'] = poster_paths + [None] * (len(data) - 10)  # Fill rest with None
print("Downloaded posters for first 10 movies")
print(data.head())

Downloaded posters for first 10 movies
      id                        title                 genre original_language  \
0    278     The Shawshank Redemption           Drama,Crime                en   
1  19404  Dilwale Dulhania Le Jayenge  Comedy,Drama,Romance                hi   
2    238                The Godfather           Drama,Crime                en   
3    424             Schindler's List     Drama,History,War                en   
4    240       The Godfather: Part II           Drama,Crime                en   

                                            overview  popularity release_date  \
0  Framed in the 1940s for the double murder of h...      94.075   1994-09-23   
1  Raj is a rich, carefree, happy-go-lucky second...      25.408   1995-10-19   
2  Spanning the years 1945 to 1955, a chronicle o...      90.585   1972-03-14   
3  The true story of how businessman Oskar Schind...      44.761   1993-12-15   
4  In the continuing saga of the Corleone crime f...      57.749   19

In [ ]:
# Feature extraction


In [39]:
# Load CNN Model
if tensorflow_available:
    base_model = ResNet50(weights='imagenet')
    model = Model(inputs=base_model.input, outputs=base_model.layers[-2].output)
    print("CNN model loaded successfully.")
else:
    model = None
    print("CNN model not loaded - TensorFlow not available.")

# Feature extraction functions
def extract_text_features(data):
    # Combine genre and overview for text features safely
    genre_text = data['genre'].fillna('').astype(str)
    overview_text = data['overview'].fillna('').astype(str)
    tags = (genre_text + ' ' + overview_text).fillna('').astype(str)

    # Keep a copy and ensure all docs are strings
    data = data.copy()
    data['tags'] = tags.apply(lambda x: x if isinstance(x, str) else str(x))
    data['tags'] = data['tags'].fillna('').astype(str)

    # Vectorize the text
    cv = CountVectorizer(max_features=5000, stop_words='english')
    vectors = cv.fit_transform(data['tags']).toarray()
    return vectors, cv

def extract_image_features(data, model):
    if model is None:
        return None
    
    features = []
    for path in data['poster_path']:
        if path and os.path.exists(path):
            img = image.load_img(path, target_size=(224, 224))
            img_array = image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0)
            img_array = preprocess_input(img_array)
            feature = model.predict(img_array)
            features.append(feature.flatten())
        else:
            features.append(np.zeros(2048))  # ResNet50 output size
    return np.array(features)

# Extract features
print("Extracting text features...")
text_features, cv = extract_text_features(data)

print("Extracting image features...")
image_features = extract_image_features(data, model)

# Combine features
if image_features is not None:
    # Normalize features
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    text_features_scaled = scaler.fit_transform(text_features)
    image_features_scaled = scaler.fit_transform(image_features)
    combined_features = np.hstack([text_features_scaled, image_features_scaled])
else:
    combined_features = text_features

print("Features extracted and combined successfully.")

# Compute similarity matrix
similarity = cosine_similarity(combined_features)
print("Similarity matrix computed.")

# Recommendation function
def recommend(movie_title):
    movie_index = data[data['title'].str.lower() == movie_title.lower()].index
    if len(movie_index) == 0:
        return "Movie not found in dataset."
    
    movie_index = movie_index[0]
    distances = similarity[movie_index]
    movie_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    recommendations = []
    for i in movie_list:
        recommendations.append(data.iloc[i[0]]['title'])
    
    return recommendations

print("Recommendation system ready!")

# Test the recommendation system
print("\nTesting recommendation system:")
test_movie = data['title'].iloc[0]  # First movie in dataset
print(f"Recommendations for '{test_movie}':")
recs = recommend(test_movie)
if isinstance(recs, list):
    for rec in recs:
        print(f"- {rec}")
else:
    print(recs)

CNN model not loaded - TensorFlow not available.
Extracting text features...
Extracting image features...
Features extracted and combined successfully.
Similarity matrix computed.
Recommendation system ready!

Testing recommendation system:
Recommendations for 'The Shawshank Redemption':
- Brubaker
- The Woodsman
- Cool Hand Luke
- A Prophet
- The Getaway


In [40]:
# performance evaluation (using a simple precision/recall approach based on genres)

# ground truth: movies with same genre are relevant
ground_truth = { 
    "The Shawshank Redemption": ["Drama"],
    "The Godfather": ["Crime", "Drama"],
}

In [41]:
# Precision and recall calculation

def precision_k(recommended_genre_sets, relevant, k):
    hits = sum(1 for genres in recommended_genre_sets if genres.intersection(relevant))
    return hits / k

def recall_k(recommended_genre_sets, relevant, k):
    if len(relevant) == 0:
        return 0.0
    found_genres = set().union(*[genres for genres in recommended_genre_sets if genres.intersection(relevant)])
    return len(found_genres.intersection(relevant)) / len(relevant)

In [42]:
# Evaluation Function
def evaluate_system(movie_title, k=5):
    movie_index = data[data['title'].str.lower() == movie_title.lower()].index[0]
    distance = sorted(list(enumerate(similarity[movie_index])), reverse=True, key=lambda x: x[1])[1:]
    recommendation = []
    recommended_genre_sets = []
    for i in distance[:k]:
        title = data.iloc[i[0]]['title']
        recommendation.append(title)
        genres = data.iloc[i[0]]['genre']
        genre_set = set([g.strip() for g in str(genres).split(',') if g.strip()])
        recommended_genre_sets.append(genre_set)
    relevant = set(ground_truth.get(movie_title, []))
    
    precision = precision_k(recommended_genre_sets, relevant, k)
    recall = recall_k(recommended_genre_sets, relevant, k)

    print("Evaluation Results:")
    print("\nmovie_title: ", movie_title)
    print("Recommendation:")
    for rec in recommendation:
        print(f"- {rec}")

    print(f"Precision@{k}: {precision:.2f}")
    print(f"Recall@{k}: {recall:.2f}")

In [43]:
# run evaluation for test movies
evaluate_system("The Shawshank Redemption")
evaluate_system("The Godfather")

Evaluation Results:

movie_title:  The Shawshank Redemption
Recommendation:
- Brubaker
- The Woodsman
- Cool Hand Luke
- A Prophet
- The Getaway
Precision@5: 1.00
Recall@5: 1.00


Evaluation Results:

movie_title:  The Godfather
Recommendation:
- The Godfather: Part II
- Blood Ties
- Bomb City
- Joker
- Felon
Precision@5: 1.00
Recall@5: 1.00
